# C11-neural-training — Practice p22 — Solution


**Type:** challenge · **Difficulty:** advanced · **Concepts:** manual-backpropagation


The exact forward ledger (column-vector shapes) is
\(z_1=(3,1,-6)^T\;(3,1)\), \(h_1=(3,1,0)^T\;(3,1)\),
\(z_2=(6,-6)^T\;(2,1)\), \(h_2=(6,0)^T\;(2,1)\),
\(\hat y=25/2\;(1,1)\), and \(L=361/8\;(1,1)\).

Backward, \(d\hat y=19/2\;(1,1)\),
\(dw_3=(57,0)^T\;(2,1)\), \(db_3=19/2\;(1,1)\),
\(dh_2=(19,-19/2)^T\;(2,1)\), and after the ReLU gate
\(dz_2=(19,0)^T\;(2,1)\). Then
\(dW_2=[[57,19,0],[0,0,0]]\;(2,3)\),
\(db_2=(19,0)^T\;(2,1)\),
\(dh_1=(19,38,-19)^T\;(3,1)\),
\(dz_1=(19,38,0)^T\;(3,1)\),
\(dW_1=[[19,-38],[38,-76],[0,0]]\;(3,2)\),
\(db_1=(19,38,0)^T\;(3,1)\), and \(dx=(95,19)^T\;(2,1)\).

Each \(dh\) is a matrix-transpose product that sums contributions from all
downstream coordinates before the elementwise ReLU mask is applied. For one
example, the consistency identities are \(dW_2=dz_2h_1^T\),
\(dW_1=dz_1x^T\), and each bias gradient equals its preactivation gradient.


In [ ]:
from fractions import Fraction as F
x_p22=[[F(1)],[F(-2)]]; z1_p22=[[F(3)],[F(1)],[F(-6)]]; h1_p22=[[F(3)],[F(1)],[F(0)]]
z2_p22=[[F(6)],[F(-6)]]; h2_p22=[[F(6)],[F(0)]]; yhat_p22=F(25,2); loss_p22=F(361,8)
dy_p22=F(19,2); dw3_p22=[[F(57)],[F(0)]]; db3_p22=F(19,2); dh2_p22=[[F(19)],[F(-19,2)]]
dz2_p22=[[F(19)],[F(0)]]; dW2_p22=[[F(57),F(19),F(0)],[F(0),F(0),F(0)]]; db2_p22=dz2_p22
dh1_p22=[[F(19)],[F(38)],[F(-19)]]; dz1_p22=[[F(19)],[F(38)],[F(0)]]
dW1_p22=[[F(19),F(-38)],[F(38),F(-76)],[F(0),F(0)]]; db1_p22=dz1_p22; dx_p22=[[F(95)],[F(19)]]


### Answer check


In [ ]:
# Independently rebuild the entire exact graph from the statement constants.
W1_check_p22 = [[F(1), F(-1)], [F(2), F(1)], [F(-1), F(2)]]
b1_check_p22 = [[F(0)], [F(1)], [F(-1)]]
W2_check_p22 = [[F(1), F(2), F(-1)], [F(-2), F(1), F(1)]]
b2_check_p22 = [[F(1)], [F(-1)]]
w3_check_p22 = [[F(2)], [F(-1)]]
b3_check_p22 = F(1, 2)

def matvec_p22(matrix, vector):
    return [[sum(matrix[i][j] * vector[j][0] for j in range(len(vector)))] for i in range(len(matrix))]

def transpose_matvec_p22(matrix, vector):
    return [[sum(matrix[i][j] * vector[i][0] for i in range(len(matrix)))] for j in range(len(matrix[0]))]

def addvec_p22(left, right):
    return [[a[0] + b[0]] for a, b in zip(left, right)]

def outer_p22(left, right):
    return [[left[i][0] * right[j][0] for j in range(len(right))] for i in range(len(left))]

z1_check_p22 = addvec_p22(matvec_p22(W1_check_p22, x_p22), b1_check_p22)
h1_check_p22 = [[max(F(0), value[0])] for value in z1_check_p22]
z2_check_p22 = addvec_p22(matvec_p22(W2_check_p22, h1_check_p22), b2_check_p22)
h2_check_p22 = [[max(F(0), value[0])] for value in z2_check_p22]
yhat_check_p22 = sum(w3_check_p22[i][0] * h2_check_p22[i][0] for i in range(2)) + b3_check_p22
loss_check_p22 = F(1, 2) * (yhat_check_p22 - F(3)) ** 2
dy_check_p22 = yhat_check_p22 - F(3)
dw3_check_p22 = [[dy_check_p22 * value[0]] for value in h2_check_p22]
db3_check_p22 = dy_check_p22
dh2_check_p22 = [[dy_check_p22 * value[0]] for value in w3_check_p22]
dz2_check_p22 = [[dh2_check_p22[i][0] if z2_check_p22[i][0] > 0 else F(0)] for i in range(2)]
dW2_check_p22 = outer_p22(dz2_check_p22, h1_check_p22)
db2_check_p22 = dz2_check_p22
dh1_check_p22 = transpose_matvec_p22(W2_check_p22, dz2_check_p22)
dz1_check_p22 = [[dh1_check_p22[i][0] if z1_check_p22[i][0] > 0 else F(0)] for i in range(3)]
dW1_check_p22 = outer_p22(dz1_check_p22, x_p22)
db1_check_p22 = dz1_check_p22
dx_check_p22 = transpose_matvec_p22(W1_check_p22, dz1_check_p22)

assert z1_check_p22 == z1_p22 and h1_check_p22 == h1_p22
assert z2_check_p22 == z2_p22 and h2_check_p22 == h2_p22
assert yhat_check_p22 == yhat_p22 == F(25, 2)
assert loss_check_p22 == loss_p22 == F(361, 8)
assert dy_check_p22 == dy_p22 and dw3_check_p22 == dw3_p22 and db3_check_p22 == db3_p22
assert dh2_check_p22 == dh2_p22 and dz2_check_p22 == dz2_p22
assert dW2_check_p22 == dW2_p22 and db2_check_p22 == db2_p22
assert dh1_check_p22 == dh1_p22 and dz1_check_p22 == dz1_p22
assert dW1_check_p22 == dW1_p22 and db1_check_p22 == db1_p22
assert dx_check_p22 == dx_p22 == [[F(95)], [F(19)]]
assert [len(dW2_p22), len(dW2_p22[0]), len(dW1_p22), len(dW1_p22[0]), len(dx_p22)] == [2, 3, 3, 2, 2]
